<a href="https://colab.research.google.com/github/el07n/Deep-learning/blob/main/notebooks/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SmartPlate AI - EfficientNetV2B0 Training and Evaluation

This notebook builds the ImageNet-pretrained EfficientNetV2B0 multi-task model, trains the new heads, fine-tunes the final backbone layers, evaluates the official test split, and saves the final artifacts to Google Drive.

Before running, select **Runtime > Change runtime type > T4 GPU**.

## 1. Confirm that Colab has a GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow version:", tf.__version__)
print("Visible GPUs:", gpus)
assert gpus, "No GPU detected. Select Runtime > Change runtime type > T4 GPU."

## 2. Download the project and install its dependencies

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/el07n/Deep-learning.git"
PROJECT_DIR = Path("/content/Deep-learning")

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(f"Project ready at: {PROJECT_DIR}")

## 3. Mount Google Drive for the final model artifacts
The model and evaluation results are saved permanently under `MyDrive/SmartPlateAI/artifacts`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ARTIFACTS_DIR = Path("/content/drive/MyDrive/SmartPlateAI/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print("Artifacts will be saved to:", ARTIFACTS_DIR)

## 4. Prepare Nutrition5k if it is not already available
This makes the notebook standalone. It downloads only the official overhead RGB subset and applies the same cleaning pipeline used in the data-preparation notebook.

In [ ]:
DATASET_ROOT = Path("/content/nutrition5k_dataset")
PREPARED_DIR = Path("/content/smartplate_processed")

if not (PREPARED_DIR / "manifest.csv").exists():
    subprocess.run([
        sys.executable, "-m", "scripts.download_nutrition5k_support_files",
        "--dataset-root", str(DATASET_ROOT),
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "scripts.download_nutrition5k_subset",
        "--dataset-root", str(DATASET_ROOT),
        "--workers", "16",
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "scripts.prepare_nutrition5k",
        "--dataset-root", str(DATASET_ROOT),
        "--output-dir", str(PREPARED_DIR),
        "--max-ingredients", "50",
        "--exclude-ids", "data/exclude_dish_ids.txt",
    ], check=True)
else:
    print("Prepared manifest already exists; skipping download and cleaning.")

assert (PREPARED_DIR / "manifest.csv").exists()
print("Prepared data ready at:", PREPARED_DIR)

## 5. Inspect the actual model-building code
The main component is Keras EfficientNetV2B0 with official ImageNet weights. A shared feature layer feeds a 50-label sigmoid ingredient head and a two-value softplus nutrition head.

In [ ]:
import inspect
from smartplate.model import build_multitask_model, unfreeze_for_fine_tuning

print(inspect.getsource(build_multitask_model))
print(inspect.getsource(unfreeze_for_fine_tuning))

## 6. Build the pre-trained model and display its architecture

In [ ]:
from smartplate.config import ProjectConfig

config = ProjectConfig(image_size=224, batch_size=16, max_ingredients=50)
preview_model, preview_backbone = build_multitask_model(
    num_ingredients=50,
    config=config,
    weights="imagenet",
)
preview_model.summary()
print("Backbone:", preview_backbone.name)
print("Backbone initially trainable:", preview_backbone.trainable)

## 7. Train the new heads and fine-tune EfficientNetV2B0
Stage 1 freezes the ImageNet backbone and trains the new task heads. Stage 2 unfreezes the final 25% of the backbone with a learning rate of `1e-5`.

In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.train",
    "--prepared-dir", str(PREPARED_DIR),
    "--output-dir", str(ARTIFACTS_DIR),
    "--head-epochs", "12",
    "--fine-tune-epochs", "12",
    "--batch-size", "16",
], check=True)

print("Training and fine-tuning completed.")

## 8. Evaluate the official test split and calibrate uncertainty

In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.evaluate",
    "--prepared-dir", str(PREPARED_DIR),
    "--artifacts-dir", str(ARTIFACTS_DIR),
    "--threshold", "0.50",
    "--coverage", "0.95",
], check=True)

print("Evaluation completed.")

## 9. Display the final test metrics

In [ ]:
import json
import pandas as pd
from IPython.display import display

metrics = json.loads((ARTIFACTS_DIR / "evaluation_metrics.json").read_text(encoding="utf-8"))
results = pd.DataFrame({
    "Metric": [
        "Recognition Micro F1", "Recognition Macro F1",
        "Calories MAE", "Calories RMSE", "Calories R2",
        "Protein MAE", "Protein RMSE", "Protein R2",
    ],
    "Result": [
        metrics["recognition"]["micro_f1"],
        metrics["recognition"]["macro_f1"],
        metrics["nutrition"]["calories"]["mae"],
        metrics["nutrition"]["calories"]["rmse"],
        metrics["nutrition"]["calories"]["r2"],
        metrics["nutrition"]["protein"]["mae"],
        metrics["nutrition"]["protein"]["rmse"],
        metrics["nutrition"]["protein"]["r2"],
    ],
})
display(results.style.format({"Result": "{:.4f}"}))

## 10. Display predicted versus actual nutrition

In [ ]:
from IPython.display import Image as DisplayImage

display(DisplayImage(filename=str(ARTIFACTS_DIR / "predicted_vs_actual.png")))

## 11. Verify the saved artifacts

In [ ]:
required = [
    "smartplate.keras",
    "ingredient_vocabulary.json",
    "target_scaler.json",
    "project_config.json",
    "evaluation_metrics.json",
    "uncertainty.json",
]

for name in required:
    path = ARTIFACTS_DIR / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

assert all((ARTIFACTS_DIR / name).exists() for name in required)
print("\nAll final model artifacts are ready in Google Drive.")